# Olist Distribution Analysis

목적: Olist 데이터를 실제 5PL 시장 데이터로 직접 사용하는 것이 아니라,
시뮬레이션 입력 분포를 설정하기 위한 empirical proxy로 활용한다.

분석 대상:
1. 주문당 SKU 수
2. SKU별 주문수량
3. Seller별 취급 SKU 수
4. Seller-SKU별 거래규모
5. 상품 가격
6. 배송비

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")

ITEMS_PATH = DATA_DIR / "olist_order_items_dataset.csv"
ORDERS_PATH = DATA_DIR / "olist_orders_dataset.csv"
PRODUCTS_PATH = DATA_DIR / "olist_products_dataset.csv"
SELLERS_PATH = DATA_DIR / "olist_sellers_dataset.csv"

In [3]:
items = pd.read_csv(
    ITEMS_PATH,
    usecols=[
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
)

orders = pd.read_csv(
    ORDERS_PATH,
    usecols=[
        "order_id",
        "order_status",
        "order_purchase_timestamp"
    ]
)

products = pd.read_csv(
    PRODUCTS_PATH,
    usecols=[
        "product_id",
        "product_category_name"
    ]
)

sellers = pd.read_csv(
    SELLERS_PATH,
    usecols=[
        "seller_id",
        "seller_state"
    ]
)

In [4]:
print("items:", items.shape)
print("orders:", orders.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)

items: (112650, 6)
orders: (99441, 3)
products: (32951, 2)
sellers: (3095, 2)


In [5]:
display(items.head())
display(orders.head())
display(products.head())
display(sellers.head())

,order_id,order_item_id,product_id,seller_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.90,18.14


,order_id,order_status,order_purchase_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39


,product_id,product_category_name
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria
1,3aa071139cb16b67ca9e5dea641aaa2f,artes
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer
3,cef67bcfe19066a932b7673e239eb23d,bebes
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas


,seller_id,seller_state
0,3442f8959a84dea7ee197c632cb2df15,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,SP


In [6]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_id       112650 non-null  object 
 1   order_item_id  112650 non-null  int64  
 2   product_id     112650 non-null  object 
 3   seller_id      112650 non-null  object 
 4   price          112650 non-null  float64
 5   freight_value  112650 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 5.2+ MB


In [7]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   order_id                  99441 non-null  object
 1   order_status              99441 non-null  object
 2   order_purchase_timestamp  99441 non-null  object
dtypes: object(3)
memory usage: 2.3+ MB


In [8]:
print("ITEMS")
display(items.isna().sum())

print("ORDERS")
display(orders.isna().sum())

print("PRODUCTS")
display(products.isna().sum())

print("SELLERS")
display(sellers.isna().sum())

ITEMS


order_id         0
order_item_id    0
product_id       0
seller_id        0
price            0
freight_value    0
dtype: int64

ORDERS


order_id                    0
order_status                0
order_purchase_timestamp    0
dtype: int64

PRODUCTS


product_id                 0
product_category_name    610
dtype: int64

SELLERS


seller_id       0
seller_state    0
dtype: int64

### 주문 시점을 날짜형으로 변환
Seller-SKU-월별 거래규모를 계산할 때 필요

In [9]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_purchase_timestamp"].describe()

count                            99441
mean     2017-12-31 08:43:12.776581120
min                2016-09-04 21:15:19
25%                2017-09-12 14:46:19
50%                2018-01-18 23:04:36
75%                2018-05-04 15:42:16
max                2018-10-17 17:30:18
Name: order_purchase_timestamp, dtype: object

In [10]:
orders["year_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

### 실제 완료 주문만 남길지 확인

In [11]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64